# 02 Feature Engineering — Yelp Fake Review Dataset

## Goal
Create leakage-safe, realistic behavioral and lightweight text features for fake review detection.

## Important Rule
We will only create features from observable behavior, not direct fraud-score formulas.

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv('../data/cleaned/train_cleaned.csv')
test_df = pd.read_csv('../data/cleaned/test_cleaned.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (9929, 22)
Test shape: (2483, 22)


In [3]:
train_df['date'] = pd.to_datetime(train_df['date'])
train_df['yelpJoinDate'] = pd.to_datetime(train_df['yelpJoinDate'])

test_df['date'] = pd.to_datetime(test_df['date'])
test_df['yelpJoinDate'] = pd.to_datetime(test_df['yelpJoinDate'])

In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9929 entries, 0 to 9928
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   reviewID           9929 non-null   str           
 1   reviewerID         9929 non-null   str           
 2   restaurantID       9929 non-null   str           
 3   date               9929 non-null   datetime64[us]
 4   rating             9929 non-null   int64         
 5   reviewUsefulCount  9929 non-null   int64         
 6   reviewContent      9926 non-null   str           
 7   flagged            9929 non-null   int64         
 8   name               9929 non-null   str           
 9   location           9929 non-null   str           
 10  yelpJoinDate       9929 non-null   datetime64[us]
 11  friendCount        9929 non-null   int64         
 12  reviewCount        9929 non-null   int64         
 13  firstCount         9929 non-null   int64         
 14  usefulCount        

In [5]:
train_df['reviewer_account_age_days'] = (
    train_df['date'] - train_df['yelpJoinDate']
).dt.days

test_df['reviewer_account_age_days'] = (
    test_df['date'] - test_df['yelpJoinDate']
).dt.days

In [6]:
train_df['reviewer_account_age_days'].describe()

count    9929.000000
mean      311.839964
std       411.275166
min         0.000000
25%        20.000000
50%       100.000000
75%       487.000000
max      2395.000000
Name: reviewer_account_age_days, dtype: float64

In [7]:
train_df.groupby('flagged')['reviewer_account_age_days'].mean()

flagged
0    490.563476
1    130.979737
Name: reviewer_account_age_days, dtype: float64

In [8]:
train_df['reviewer_engagement_ratio'] = (
    train_df['usefulCount'] +
    train_df['coolCount'] +
    train_df['funnyCount']
) / (train_df['reviewCount'] + 1)

test_df['reviewer_engagement_ratio'] = (
    test_df['usefulCount'] +
    test_df['coolCount'] +
    test_df['funnyCount']
) / (test_df['reviewCount'] + 1)

In [9]:
train_df['reviewer_engagement_ratio'].describe()

count    9929.000000
mean        1.403871
std         2.639714
min         0.000000
25%         0.181818
50%         0.750000
75%         1.640884
max        80.000000
Name: reviewer_engagement_ratio, dtype: float64

In [10]:
train_df.groupby('flagged')['reviewer_engagement_ratio'].mean()

flagged
0    2.244437
1    0.553256
Name: reviewer_engagement_ratio, dtype: float64

In [11]:
train_df['reviewer_social_reach'] = (
    train_df['friendCount'] +
    train_df['fanCount']
)

test_df['reviewer_social_reach'] = (
    test_df['friendCount'] +
    test_df['fanCount']
)

In [12]:
train_df['reviewer_social_reach'].describe()

count    9929.000000
mean       35.216739
std       182.812884
min         0.000000
25%         0.000000
50%         1.000000
75%        14.000000
max      4448.000000
Name: reviewer_social_reach, dtype: float64

In [13]:
train_df.groupby('flagged')['reviewer_social_reach'].mean()

flagged
0    67.395675
1     2.653090
Name: reviewer_social_reach, dtype: float64

In [15]:
train_df['reviewContent'] = train_df['reviewContent'].fillna('').astype(str)
test_df['reviewContent'] = test_df['reviewContent'].fillna('').astype(str)

In [16]:
train_df['uppercase_ratio'] = train_df['reviewContent'].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)

test_df['uppercase_ratio'] = test_df['reviewContent'].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)

In [17]:
train_df['uppercase_ratio'].describe()

count    9929.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: uppercase_ratio, dtype: float64

In [18]:
train_df.groupby('flagged')['uppercase_ratio'].mean()

flagged
0    0.0
1    0.0
Name: uppercase_ratio, dtype: float64

In [19]:
feature_tracker = [
    {
        "feature_name": "reviewer_account_age_days",
        "feature_type": "Time-Based Feature",
        "source_columns": "date, yelpJoinDate",
        "purpose": "Measures how old the reviewer account was at review time",
        "leakage_risk": "Low",
        "status": "Created",
        "notes": "Safe behavioral feature; genuine reviewers have older accounts on average"
    },
    {
        "feature_name": "reviewer_engagement_ratio",
        "feature_type": "Reviewer Reputation Feature",
        "source_columns": "usefulCount, coolCount, funnyCount, reviewCount",
        "purpose": "Measures community engagement received per review activity",
        "leakage_risk": "Low",
        "status": "Created",
        "notes": "Safe ratio feature; avoids direct fraud scoring"
    },
    {
        "feature_name": "reviewer_social_reach",
        "feature_type": "Reviewer Social Feature",
        "source_columns": "friendCount, fanCount",
        "purpose": "Measures reviewer social presence and platform credibility",
        "leakage_risk": "Low",
        "status": "Created",
        "notes": "Observable reviewer reputation signal"
    },
    {
        "feature_name": "uppercase_ratio",
        "feature_type": "Lightweight Text Feature",
        "source_columns": "reviewContent",
        "purpose": "Measures capitalization behavior in review text",
        "leakage_risk": "Low",
        "status": "Dropped",
        "notes": "Not useful because reviewContent was already lowercased during cleaning, resulting in all zero values"
    }
]

feature_tracker_df = pd.DataFrame(feature_tracker)
feature_tracker_df

,feature_name,feature_type,source_columns,purpose,leakage_risk,status,notes
0,reviewer_account_age_days,Time-Based Feature,"date, yelpJoinDate",Measures how old the reviewer account was at r...,Low,Created,Safe behavioral feature; genuine reviewers hav...
1,reviewer_engagement_ratio,Reviewer Reputation Feature,"usefulCount, coolCount, funnyCount, reviewCount",Measures community engagement received per rev...,Low,Created,Safe ratio feature; avoids direct fraud scoring
2,reviewer_social_reach,Reviewer Social Feature,"friendCount, fanCount",Measures reviewer social presence and platform...,Low,Created,Observable reviewer reputation signal
3,uppercase_ratio,Lightweight Text Feature,reviewContent,Measures capitalization behavior in review text,Low,Dropped,Not useful because reviewContent was already l...


In [20]:
train_df['review_word_count'] = train_df['reviewContent'].apply(lambda x: len(x.split()))
test_df['review_word_count'] = test_df['reviewContent'].apply(lambda x: len(x.split()))

In [21]:
train_df['review_word_count'].describe()

count    9929.000000
mean       75.270722
std        69.646123
min         0.000000
25%        30.000000
50%        54.000000
75%        99.000000
max       593.000000
Name: review_word_count, dtype: float64

In [22]:
train_df.groupby('flagged')['review_word_count'].mean()

flagged
0    89.287745
1    61.086120
Name: review_word_count, dtype: float64

In [23]:
feature_tracker.append({
    "feature_name": "review_word_count",
    "feature_type": "Lightweight Text Feature",
    "source_columns": "reviewContent",
    "purpose": "Measures review verbosity using total word count",
    "leakage_risk": "Low",
    "status": "Created",
    "notes": (
        "Strong realistic behavioral text feature. "
        "Fake reviews tend to be shorter and lower effort. "
        "Will likely replace original ReviewLength feature to avoid redundancy."
    )
})

In [24]:
train_df['punctuation_count'] = train_df['reviewContent'].apply(
    lambda x: sum(1 for c in x if c in '!?')
)

test_df['punctuation_count'] = test_df['reviewContent'].apply(
    lambda x: sum(1 for c in x if c in '!?')
)

In [25]:
train_df['punctuation_count'].describe()

count    9929.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: punctuation_count, dtype: float64

In [26]:
train_df.groupby('flagged')['punctuation_count'].mean()

flagged
0    0.0
1    0.0
Name: punctuation_count, dtype: float64

In [27]:
feature_tracker.append({
    "feature_name": "punctuation_count",
    "feature_type": "Lightweight Text Feature",
    "source_columns": "reviewContent",
    "purpose": "Measures excessive punctuation behavior",
    "leakage_risk": "Low",
    "status": "Dropped",
    "notes": (
        "Dataset text was already pre-cleaned and punctuation removed, "
        "resulting in all zero values."
    )
})

In [28]:
train_df['reviewer_helpfulness_ratio'] = (
    train_df['usefulCount']
) / (train_df['reviewCount'] + 1)

test_df['reviewer_helpfulness_ratio'] = (
    test_df['usefulCount']
) / (test_df['reviewCount'] + 1)

In [29]:
train_df['reviewer_helpfulness_ratio'].describe()

count    9929.000000
mean        0.723758
std         1.126854
min         0.000000
25%         0.083333
50%         0.458333
75%         0.975000
max        39.600000
Name: reviewer_helpfulness_ratio, dtype: float64

In [30]:
train_df.groupby('flagged')['reviewer_helpfulness_ratio'].mean()

flagged
0    1.117582
1    0.325226
Name: reviewer_helpfulness_ratio, dtype: float64

In [31]:
feature_tracker.append({
    "feature_name": "reviewer_helpfulness_ratio",
    "feature_type": "Reviewer Reputation Feature",
    "source_columns": "usefulCount, reviewCount",
    "purpose": "Measures how much usefulness engagement a reviewer receives relative to activity",
    "leakage_risk": "Low",
    "status": "Created",
    "notes": (
        "Strong reviewer credibility signal. "
        "Genuine reviewers tend to receive more useful feedback from community."
    )
})

In [32]:
train_df['reviewer_activity_level'] = np.log1p(
    train_df['reviewCount']
)

test_df['reviewer_activity_level'] = np.log1p(
    test_df['reviewCount']
)

In [33]:
train_df['reviewer_activity_level'].describe()

count    9929.000000
mean        2.730949
std         1.584011
min         0.693147
25%         1.386294
50%         2.397895
75%         3.761200
max         7.679251
Name: reviewer_activity_level, dtype: float64

In [34]:
train_df.groupby('flagged')['reviewer_activity_level'].mean()

flagged
0    3.776118
1    1.673285
Name: reviewer_activity_level, dtype: float64

In [35]:
train_df['rating_deviation'] = abs(
    train_df['rating'] - train_df['restaurantRating']
)

test_df['rating_deviation'] = abs(
    test_df['rating'] - test_df['restaurantRating']
)

In [36]:
train_df['rating_deviation'].describe()

count    9929.000000
mean        0.928442
std         0.815802
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         3.500000
Name: rating_deviation, dtype: float64

In [37]:
train_df.groupby('flagged')['rating_deviation'].mean()

flagged
0    0.772827
1    1.085917
Name: rating_deviation, dtype: float64

In [38]:
feature_tracker.append({
    "feature_name": "reviewer_activity_level",
    "feature_type": "Reviewer Activity Feature",
    "source_columns": "reviewCount",
    "purpose": "Measures reviewer activity intensity using logarithmic transformation",
    "leakage_risk": "Low",
    "status": "Created",
    "notes": (
        "Log-transformed reviewer activity feature. "
        "Reduces outlier dominance while preserving reviewer maturity behavior."
    )
})

In [39]:
feature_tracker.append({
    "feature_name": "rating_deviation",
    "feature_type": "Behavioral Rating Feature",
    "source_columns": "rating, restaurantRating",
    "purpose": "Measures how much reviewer rating differs from restaurant average rating",
    "leakage_risk": "Low",
    "status": "Created",
    "notes": (
        "Safe behavioral inconsistency feature. "
        "Fake reviews tend to deviate more from restaurant consensus ratings."
    )
})

In [48]:
train_df.columns

Index(['reviewID', 'reviewerID', 'restaurantID', 'date', 'rating',
       'reviewUsefulCount', 'reviewContent', 'flagged', 'name', 'location',
       'yelpJoinDate', 'friendCount', 'reviewCount', 'firstCount',
       'usefulCount', 'coolCount', 'funnyCount', 'complimentCount', 'tipCount',
       'fanCount', 'restaurantRating', 'ReviewLength',
       'reviewer_account_age_days', 'reviewer_engagement_ratio',
       'reviewer_social_reach', 'uppercase_ratio', 'review_word_count',
       'punctuation_count', 'reviewer_helpfulness_ratio',
       'reviewer_activity_level', 'rating_deviation'],
      dtype='str')

In [49]:
numeric_cols = train_df.select_dtypes(include=['int64', 'float64'])

correlation = numeric_cols.corr()['flagged'].sort_values(
    ascending=False
)

correlation

flagged                       1.000000
rating_deviation              0.191897
restaurantRating             -0.056811
rating                       -0.061171
complimentCount              -0.107471
tipCount                     -0.133562
coolCount                    -0.149429
fanCount                     -0.150925
funnyCount                   -0.151834
firstCount                   -0.160069
usefulCount                  -0.175523
reviewer_social_reach        -0.177079
friendCount                  -0.177839
ReviewLength                 -0.202470
review_word_count            -0.202470
reviewUsefulCount            -0.290866
reviewer_engagement_ratio    -0.320344
reviewCount                  -0.327862
reviewer_helpfulness_ratio   -0.351590
reviewer_account_age_days    -0.437171
reviewer_activity_level      -0.663790
uppercase_ratio                    NaN
punctuation_count                  NaN
Name: flagged, dtype: float64

In [50]:
feature_tracker_df

,feature_name,feature_type,source_columns,purpose,leakage_risk,status,notes
0,reviewer_account_age_days,Time-Based Feature,"date, yelpJoinDate",Measures how old the reviewer account was at r...,Low,Created,Safe behavioral feature; genuine reviewers hav...
1,reviewer_engagement_ratio,Reviewer Reputation Feature,"usefulCount, coolCount, funnyCount, reviewCount",Measures community engagement received per rev...,Low,Created,Safe ratio feature; avoids direct fraud scoring
2,reviewer_social_reach,Reviewer Social Feature,"friendCount, fanCount",Measures reviewer social presence and platform...,Low,Created,Observable reviewer reputation signal
3,uppercase_ratio,Lightweight Text Feature,reviewContent,Measures capitalization behavior in review text,Low,Dropped,Not useful because reviewContent was already l...


In [51]:
final_features = [

    # rating behavior
    'rating',
    'restaurantRating',
    'rating_deviation',

    # reviewer reputation
    'friendCount',
    'reviewCount',
    'reviewUsefulCount',
    'usefulCount',
    'coolCount',
    'funnyCount',
    'complimentCount',
    'tipCount',
    'fanCount',

    # engineered behavioral features
    'reviewer_account_age_days',
    'reviewer_engagement_ratio',
    'reviewer_social_reach',
    'reviewer_helpfulness_ratio',
    'reviewer_activity_level',

    # text effort feature
    'review_word_count'
]

In [52]:
X_train_final = train_df[final_features]

y_train_final = train_df['flagged']

X_test_final = test_df[final_features]

y_test_final = test_df['flagged']

In [53]:
print(X_train_final.shape)
print(X_test_final.shape)

(9929, 18)
(2483, 18)


In [54]:
train_processed = X_train_final.copy()
train_processed['flagged'] = y_train_final

test_processed = X_test_final.copy()
test_processed['flagged'] = y_test_final

In [55]:
train_processed.to_csv(
    '../data/processed/train_processed.csv',
    index=False
)

test_processed.to_csv(
    '../data/processed/test_processed.csv',
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
